# Demographie und Populationen von Exoplaneten

Das Notebook analysiert die statistische Verteilung aller bestätigten Exoplaneten aus dem NASA Exoplanet Archive. Mittels Kerndichteschätzung (KDE) wird die von Fulton et al. entdeckte Radiuslücke zwischen Supererden und Sub-Neptunen vermessen und die Masseverteilung im Masse-Radius-Diagramm mit theoretischen Zustandsgleichungen verglichen.

Welche physikalischen Prozesse formen die Verteilung von Radius, Masse und Umlaufzeit in der bekannten Exoplaneten-Population?

#### **Hintergrund**

Mit mehr als 5.500 bestätigten Exoplaneten ist die Astronomie von der Entdeckung einzelner Welten zur statistischen Demographie übergegangen. Zwei fundamentale Beobachtungsstrukturen dominieren die Diskussion über Planetenentstehung und -evolution:

1. **Die Fulton-Radiuslücke (Radius Valley):** Bei kurzperiodischen Planeten ($P < 100$ d) existiert ein auffälliges Häufigkeitsdefizit zwischen $1.5$ und $2.0\,R_\oplus$. Planeten teilen sich in zwei Klassen: dichte, atmosphärenlose **Supererden** ($R \approx 1.0 - 1.4\,R_\oplus$) und gasreiche **Sub-Neptune** ($R \approx 2.0 - 3.0\,R_\oplus$).
2. **Die Neptun-Wüste (Neptunian Desert):** Im Periodenbereich unter 4 Tagen fehlen Planeten mittlerer Masse und Größe fast vollständig. Sie überleben die intensive Sternstrahlung in Kernnähe nicht als Gasplaneten.
3. **Masse-Radius-Beziehungen:** Feste Gesteins- und Eisenkerne folgen strengen Skalierungsgesetzen, während Planeten mit leichten Wasserstoff- und Heliumhüllen bei gleicher Masse dramatisch größere Radien aufweisen.

#### **Herleitung**

**1. Photoevaporativer Atmosphärenverlust.** Ein junger Stern emittiert intensive Röntgen- und extrem ultraviolette Strahlung (XUV). Der energiebegrenzte Massenverlust einer Wasserstoffatmosphäre mit Masse $M_{\text{atm}}$ und Radius $R_p$ beträgt
$$\dot{M} = \frac{\eta \pi R_p^3 F_{\text{XUV}}}{G M_p},$$
wobei $\eta \approx 0.1$ der Wirkungsgrad ist. Integriert man diesen Verlust über die ersten $\sim 100\,\text{Ma}$, zeigt sich: Ein Gesteinskern mit $M_p \lesssim 5\,M_\oplus$ verliert seine primordiale $H_2/He$-Hülle vollständig und schrumpft auf den nackten Gesteinskern ($R \lesssim 1.5\,R_\oplus$). Kerne mit $M_p \gtrsim 8\,M_\oplus$ behalten einen Hüllenanteil von $1 - 2\,\%$ und bleiben bei $R \gtrsim 2\,R_\oplus$. Dazwischen entsteht das Fulton-Tal.

**2. Kerngetriebener Massenverlust (Core-powered mass loss).** Ein alternatives Modell benötigt keine XUV-Strahlung: Die bei der Akkretion gespeicherte Wärme des abkühlenden Gesteinskerns reicht aus, um auf Zeitskalen von Milliarden Jahren schwach gebundene Atmosphären wegzublasen. Beide Modelle sagen eine Lücke bei $1.5 - 2.0\,R_\oplus$ voraus, unterscheiden sich aber in der Feinstruktur bezüglich Sternmasse und Alter.

**3. Skalierungsgesetz für feste Planetenkerne.** Für inkompressibles Material gilt $M = \frac{4}{3}\pi \rho R^3 \implies R \propto M^{1/3}$. Reale Planetenmaterialien komprimieren sich jedoch unter dem immensen Eigengravitationsdruck. Für erdähnliche Gesteinsplaneten (Zellner, Zeng et al. 2016) folgt empirisch:
$$\frac{R}{R_\oplus} \approx \left(\frac{M}{M_\oplus}\right)^{1/3.7} \approx \left(\frac{M}{M_\oplus}\right)^{0.27}.$$

**4. Gasriesen und Entartungsdruck.** Bei Riesenplaneten oberhalb von $\approx 1\,M_{\text{Jup}}$ wird das Innere durch den quantenmechanischen Entartungsdruck der Elektronen gestützt ($P \propto \rho^{5/3}$). Dies führt zu $R \propto M^{-1/3}$: Zusätzliche Masse komprimiert den Planeten so stark, dass ein Planet mit der 3-fachen Jupitermasse kleiner ist als Jupiter selbst!

#### **Daten**

Wir fragen den Gesamtkatalog bestätigter Planeten (`table="pscomppars"`) direkt vom **NASA Exoplanet Archive** ab. Die Tabelle enthält konsolidierte Parameter mit symmetrischen und asymmetrischen Fehlerbalken.

In [ ]:
%pip install -q astroquery

In [ ]:
import os
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy.stats import gaussian_kde
from astroquery.ipac.nexsci.nasa_exoplanet_archive import NasaExoplanetArchive

ASTRODATA = Path.cwd() / "astrodata"
(ASTRODATA / "astropy").mkdir(parents=True, exist_ok=True)
os.environ["XDG_CACHE_HOME"] = str(ASTRODATA)

COLUMNS = (
    "pl_name, discoverymethod, pl_orbper, pl_rade, pl_radeerr1, pl_radeerr2, "
    "pl_bmasse, pl_bmasseerr1, pl_bmasseerr2, pl_bmassprov, pl_insol, st_teff, st_mass"
)

pop = NasaExoplanetArchive.query_criteria(table="pscomppars", select=COLUMNS).to_pandas()
print(f"✅ {len(pop)} bestätigte Exoplaneten geladen.")
print(f"  Mit gemessenem Radius: {pop['pl_rade'].notna().sum()}")
print(f"  Mit gemessener Masse:  {(pop['pl_bmassprov'] == 'Mass').sum()}")

#### **Reduktion**

##### *1. Datenbereinigung und Qualitätsfilter*

Wie filtert man ungenaue Messungen und Mindestmassen heraus, um physikalische Strukturen nicht zu verwischen?

In [ ]:
# Mittlere relative Fehler
pop["rad_relerr"] = (pop["pl_radeerr1"] - pop["pl_radeerr2"]) / 2.0 / pop["pl_rade"]
pop["mass_relerr"] = (pop["pl_bmasseerr1"] - pop["pl_bmasseerr2"]) / 2.0 / pop["pl_bmasse"]

# Sub-Stichprobe 1: Für die Radiuslücke
# Planeten mit P < 100 d, präzisen Radien (< 10 % Fehler) im Bereich 1 - 4 R_Erde
small = pop[
    (pop["pl_orbper"] < 100.0) & 
    (pop["rad_relerr"] < 0.10) & 
    pop["pl_rade"].between(1.0, 4.0)
].copy()

# Sub-Stichprobe 2: Für das Masse-Radius-Diagramm
# Nur echte Massenmessungen (kein RV-Minimum sin(i)), präzise Masse und Radius (< 30 %)
mr = pop[
    (pop["pl_bmassprov"] == "Mass") & 
    (pop["mass_relerr"] < 0.30) & 
    (pop["rad_relerr"] < 0.30)
].copy()

print(f"✅ Bereinigte Datensätze:")
print(f"  Kandidaten für Radiuslücke: {len(small)} Planeten")
print(f"  Kandidaten für Masse-Radius: {len(mr)} Planeten mit präziser M und R")

#### **Eigene Messung**

##### *2. Die Perioden-Radius-Ebene und die Neptun-Wüste*

Wie verteilen sich Exoplaneten über Perioden von Stunden bis Jahrzehnten, und wo liegt die Neptun-Wüste?

In [ ]:
fig, ax = plt.subplots(figsize=(10, 5.5))

has_r = pop["pl_rade"].notna() & pop["pl_orbper"].notna() & pop["pl_radeerr1"].notna()

for method, c, label in [("Transit", "tab:blue", "Transitmethode"), ("Radial Velocity", "tab:green", "Radialgeschwindigkeit")]:
    m = has_r & (pop["discoverymethod"] == method)
    ax.scatter(pop.loc[m, "pl_orbper"], pop.loc[m, "pl_rade"], s=4, alpha=0.45, c=c, label=label)

m_other = has_r & ~pop["discoverymethod"].isin(["Transit", "Radial Velocity"])
ax.scatter(pop.loc[m_other, "pl_orbper"], pop.loc[m_other, "pl_rade"], s=4, alpha=0.3, c="grey", label="Andere Methoden")

ax.axhspan(1.5, 2.0, color="orange", alpha=0.2, label="Radiuslücke (1.5 - 2.0 R⊕)")
ax.text(1.2, 5.0, "Neptun-Wüste\n(fehlt)", fontsize=10, color="darkred", weight="bold")

ax.set_xscale("log")
ax.set_yscale("log")
ax.set_xlim(0.2, 3e4)
ax.set_ylim(0.4, 30)
ax.set_xlabel("Umlaufzeit P [Tage]")
ax.set_ylabel("Planetenradius [R_Erde]")
ax.set_title("Perioden-Radius-Ebene der bekannten Exoplaneten")
ax.legend(loc="lower right", fontsize=8, markerscale=3)
plt.show()

##### *3. Vermessung der Fulton-Radiuslücke*

Bei welchem Radius liegt das Minimum der Planetenhäufigkeit (das 'Fulton-Tal') für kurzperiodische Planeten?

In [ ]:
logr = np.log10(small["pl_rade"])
kde = gaussian_kde(logr, bw_method=0.15)
grid = np.linspace(0, np.log10(4), 400)
dens = kde(grid)

# Suche nach lokalem Minimum im Bereich 1.3 bis 2.5 R_Erde
search_range = (grid > np.log10(1.3)) & (grid < np.log10(2.5))
r_valley = 10 ** grid[search_range][np.argmin(dens[search_range])]

fig, ax1 = plt.subplots(figsize=(9, 4.5))

ax1.hist(
    small["pl_rade"], bins=np.logspace(0, np.log10(4), 30),
    color="tab:blue", alpha=0.4, label=f"Histogramm ({len(small)} Planeten, P < 100 d, σ_R < 10 %)"
)
ax1.set_xscale("log")
ax1.set_xticks([1.0, 1.5, 2.0, 3.0, 4.0], ["1.0", "1.5", "2.0", "3.0", "4.0"])
ax1.set_xlabel("Planetenradius [R_Erde]")
ax1.set_ylabel("Anzahl Planeten")

ax2 = ax1.twinx()
ax2.plot(10 ** grid, dens, c="black", lw=2, label="Gaußsche Kerndichte (KDE)")
ax2.set_yticks([])

ax1.axvline(r_valley, color="tab:red", ls="--", lw=2, label=f"Gemessenes Tal: {r_valley:.2f} R⊕")
ax1.axvspan(1.5, 2.0, color="orange", alpha=0.15, label="Fulton et al. (2017): 1.5 - 2.0 R⊕")

ax1.set_title("Die Fulton-Radiuslücke: Supererden vs. Sub-Neptune")
ax1.legend(loc="upper right", fontsize=8)
plt.show()

print(f"✅ Vermessung der Radiuslücke:")
print(f"  Minimum der Dichte (Tal):  R_valley = {r_valley:.2f} R_Erde")
print(f"  Peak 1 (Supererden):       {10**grid[grid < np.log10(r_valley)][np.argmax(dens[grid < np.log10(r_valley)])]:.2f} R_Erde")
print(f"  Peak 2 (Sub-Neptune):      {10**grid[grid > np.log10(r_valley)][np.argmax(dens[grid > np.log10(r_valley)])]:.2f} R_Erde")

##### *4. Das Masse-Radius-Diagramm*

Welche Zusammensetzungen von reinem Gestein bis hin zu Gaswelten zeigen sich im Masse-Radius-Diagramm?

In [ ]:
fig, ax = plt.subplots(figsize=(10, 6))

sc = ax.scatter(
    mr["pl_bmasse"], mr["pl_rade"],
    c=np.log10(mr["pl_insol"].where(mr["pl_insol"] > 0)),
    cmap="plasma", s=9, alpha=0.7, label=f"{len(mr)} Planeten (σ < 30 %)"
)
cbar = fig.colorbar(sc, ax=ax)
cbar.set_label("Einstrahlung log10(S / S_Erde)")

# Sonnensystem-Referenzen
sol = [("Erde", 1.0, 1.0), ("Neptun", 17.15, 3.88), ("Saturn", 95.16, 9.45), ("Jupiter", 317.8, 11.21)]
for name, me, re in sol:
    ax.scatter([me], [re], marker="o", s=70, facecolors="none", edgecolors="black", lw=1.5)
    ax.text(me * 1.15, re * 0.9, name, fontsize=8, weight="bold")

ax.set_xscale("log")
ax.set_yscale("log")
ax.set_xlim(0.1, 1e4)
ax.set_ylim(0.3, 30)
ax.set_xlabel("Masse [M_Erde]")
ax.set_ylabel("Radius [R_Erde]")
ax.set_title("Masse-Radius-Verteilung bekannter Exoplaneten")
ax.legend(loc="upper left", fontsize=8)
plt.show()

#### **Modell und Fit**

##### *5. Theoretische Dichtekurven und Gesteinsrelation*

Welche physikalischen Zustandsgleichungen beschreiben die beobachteten Planetenradien?

In [ ]:
fig, ax = plt.subplots(figsize=(10, 6))

ax.scatter(mr["pl_bmasse"], mr["pl_rade"], s=5, c="lightgrey", alpha=0.5, label="Beobachtungen")

M_grid = np.logspace(-1, 4, 300)

# Linien konstanter Dichte (rho in g/cm³)
for rho, ls, label in [(1.0, ":", "ρ = 1.0 g/cm³ (Wasser/Gas)"), 
                       (5.51, "--", "ρ = 5.51 g/cm³ (Erde / Gestein)"), 
                       (10.0, "-.", "ρ = 10.0 g/cm³ (Eisen)")]:
    R_grid = (M_grid * 5.972e27 / (4 / 3 * np.pi * rho)) ** (1 / 3) / 6.371e8
    ax.plot(M_grid, R_grid, c="tab:gray", ls=ls, lw=1.2, label=label)

# Zeng et al. (2016) empirische Kurve für reine Silikat-Gesteinsplaneten
M_rock = np.logspace(-1, np.log10(20), 100)
R_rock = 1.0 * M_rock ** (1 / 3.7)
ax.plot(M_rock, R_rock, c="saddlebrown", lw=2.2, label="Reines Gestein (Zeng et al. 2016)")

for name, me, re in sol:
    ax.scatter([me], [re], marker="o", s=60, facecolors="none", edgecolors="black", lw=1.5)
    ax.text(me * 1.15, re * 0.9, name, fontsize=8)

ax.set_xscale("log")
ax.set_yscale("log")
ax.set_xlim(0.1, 1e4)
ax.set_ylim(0.3, 30)
ax.set_xlabel("Masse [M_Erde]")
ax.set_ylabel("Radius [R_Erde]")
ax.set_title("Masse-Radius-Diagramm mit physikalischen Zustandsgleichungen")
ax.legend(loc="upper left", fontsize=8)
plt.show()

#### **Eigene Messung gegen den Katalog**

##### *6. Vergleich mit den Entdeckungsarbeiten von Fulton et al. (2017)*

Wie stimmt unsere gemessene Tal-Position mit den veröffentlichten Literaturwerten überein?

In [ ]:
lit_valleys = [
    ("Fulton et al. (2017) [Kepler California Survey]", 1.75, "1.5 - 2.0"),
    ("Van Eylen et al. (2018) [Asteroseismology Sample]", 1.74, "1.74 ± 0.05"),
    ("Martinez et al. (2019) [CKS Gaia DR2 Sample]", 1.72, "1.72 ± 0.04"),
]

print(f"{'Publikation / Quelle':<45} {'Tal-Minimum [R_Erde]':<22} {'Bereich':<15}")
print("-" * 82)
print(f"{'Eigene Messung (NASA Exoplanet Archive)':<45} {r_valley:<22.2f} {'1.5 - 2.0'}")
for pub, val, rng in lit_valleys:
    print(f"{pub:<45} {val:<22.2f} {rng:<15}")

#### **Variationen**

Jeweils zuerst die Vorhersage notieren, dann den Code anpassen und testen:

1. **Abhängigkeit von der Sternmasse:** Teile die Sub-Stichprobe in kühle Sterne ($M_\star < 0.8\,M_\odot$) und sonnenähnliche Sterne ($M_\star \ge 1.0\,M_\odot$). Wandert die Radiuslücke bei masseärmeren Sternen zu kleineren Radien? *(Vorhersage: Ja, leichtere Sterne emittieren weniger XUV-Gesamtenergie, sodass die Photoevaporation bei kleineren Kernmassen stoppt).*
2. **KDE-Bandbreite:** Ändere `bw_method` von 0.15 auf 0.05 bzw. 0.30. Wie robust ist die Position des Minimums gegen Überschärfung und Glättung?
3. **Insolationsfilter:** Beschränke die Analyse auf stark eingestrahlte Planeten ($S > 100\,S_\oplus$). Was geschieht mit dem Peak der Sub-Neptune?

#### **Nächste Schritte**

So würde eine populationsstatistische Studie vertieft:
- **Completeness Correction:** Berechnung der geometrischen Transitwahrscheinlichkeit $P_{\text{tr}} = R_\star / a$ und der detektionsspezifischen Pipeline-Effizienz zur Bestimmung wahrer intrinsischer Vorkommensraten (Occurrence Rates per Star).
- **Zwei-dimensionale Valley-Steigung:** Modellierung der Radiuslücke als Funktion der Periode $R_{\text{valley}}(P) = R_0 (P / P_0)^m$. Die Steigung $m$ unterscheidet Photoevaporation ($m \approx -0.15$) von kerngetriebenem Massenverlust ($m \approx -0.11$).
- **Interne Strukturmodelle:** Inferenz der Hüllenmassenfraktionen $f_{\text{atm}} = M_{\text{atm}} / M_{\text{core}}$ über thermodynamische M-R-Gitter (z. B. Lopez & Fortney 2014).

#### **Referenzen**

- Fulton, B. J. et al. (2017): *The California-Kepler Survey. III. A Gap in the Radius Distribution of Small Planets*, AJ, 154, 109.
- Owen, J. E. & Wu, Y. (2017): *Evaporating Exoplanets. II. UV-driven Ground States*, ApJ, 847, 29.
- Ginzburg, S., Schlichting, H. E. & Sari, R. (2018): *Core-powered mass loss and the radius valley*, MNRAS, 476, 759–765.
- Zeng, L., Sasselov, D. D. & Jacobsen, S. B. (2016): *Mass-Radius Relation for Compact Exoplanets*, ApJ, 819, 127.
- Van Eylen, V. et al. (2018): *An asteroseismic view of the radius valley: stripped cores, not water worlds*, MNRAS, 479, 4786–4795.